In [7]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import requests
from io import BytesIO
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import numpy as np

# Configuration
CSV_PATH = 'matched_SMALL.csv'
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cpu


In [12]:
class ImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        url = self.dataframe.iloc[idx]['image_url']
        label = int(self.dataframe.iloc[idx]['high_reorder'])
        
        try:
            # Fetch image from URL
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status() # Check for HTTP errors
            image = Image.open(BytesIO(response.content)).convert('RGB')
        except Exception as e:
            # Fallback for failed downloads or broken links
            # Returns a neutral gray image so the batch doesn't crash
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color=(128, 128, 128))
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.float32)

# Preprocessing Pipeline (same as before)
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load and Split Data
df = pd.read_csv(CSV_PATH)
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42)

train_loader = DataLoader(ImageDataset(train_df, transform), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ImageDataset(val_df, transform), batch_size=BATCH_SIZE)
test_loader = DataLoader(ImageDataset(test_df, transform), batch_size=BATCH_SIZE)

print(f"Dataset Split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

Dataset Split: Train=94, Val=20, Test=21


In [13]:
def train_model(model, criterion, optimizer, num_epochs=EPOCHS):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE).view(-1, 1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")

def evaluate_model(model):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE).view(-1, 1)
            outputs = torch.sigmoid(model(inputs))
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy().astype(int)) # Force labels to int here

    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    
    print(f"\n--- Evaluation Results ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"ROC-AUC:  {roc:.4f}")
    print(f"Confusion Matrix:\n{cm}")

In [14]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

model_cnn = SimpleCNN().to(DEVICE)
optimizer = optim.Adam(model_cnn.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()

print("Training Custom CNN...")
train_model(model_cnn, criterion, optimizer)
evaluate_model(model_cnn)

Training Custom CNN...
Epoch 1/10, Loss: 4.2022
Epoch 2/10, Loss: 3.0535
Epoch 3/10, Loss: 0.8628
Epoch 4/10, Loss: 0.6521
Epoch 5/10, Loss: 0.6372
Epoch 6/10, Loss: 0.6140
Epoch 7/10, Loss: 0.5848
Epoch 8/10, Loss: 0.5432
Epoch 9/10, Loss: 0.4940
Epoch 10/10, Loss: 0.4427

--- Evaluation Results ---
Accuracy: 0.6667
F1 Score: 0.7742
ROC-AUC:  0.6759
Confusion Matrix:
[[ 2  7]
 [ 0 12]]


In [15]:
model_rn18_scratch = models.resnet18(weights=None)
model_rn18_scratch.fc = nn.Linear(model_rn18_scratch.fc.in_features, 1)
model_rn18_scratch = model_rn18_scratch.to(DEVICE)

optimizer = optim.Adam(model_rn18_scratch.parameters(), lr=LEARNING_RATE)
print("Training ResNet-18 from scratch...")
train_model(model_rn18_scratch, criterion, optimizer)
evaluate_model(model_rn18_scratch)

Training ResNet-18 from scratch...
Epoch 1/10, Loss: 1.3177
Epoch 2/10, Loss: 1.2846
Epoch 3/10, Loss: 0.9759
Epoch 4/10, Loss: 0.6795
Epoch 5/10, Loss: 0.5981
Epoch 6/10, Loss: 0.5695
Epoch 7/10, Loss: 0.5451
Epoch 8/10, Loss: 0.4695
Epoch 9/10, Loss: 0.4161
Epoch 10/10, Loss: 0.3512

--- Evaluation Results ---
Accuracy: 0.5238
F1 Score: 0.6875
ROC-AUC:  0.3889
Confusion Matrix:
[[ 0  9]
 [ 1 11]]


In [16]:
model_rn18_frozen = models.resnet18(weights='DEFAULT')
# Freeze all layers
for param in model_rn18_frozen.parameters():
    param.requires_grad = False

# Replace head (new layers are trainable by default)
model_rn18_frozen.fc = nn.Linear(model_rn18_frozen.fc.in_features, 1)
model_rn18_frozen = model_rn18_frozen.to(DEVICE)

optimizer = optim.Adam(model_rn18_frozen.fc.parameters(), lr=LEARNING_RATE)
print("Training Pretrained ResNet-18 (Frozen)...")
train_model(model_rn18_frozen, criterion, optimizer)
evaluate_model(model_rn18_frozen)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Colton/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


Training Pretrained ResNet-18 (Frozen)...
Epoch 1/10, Loss: 0.6908
Epoch 2/10, Loss: 0.6644
Epoch 3/10, Loss: 0.6448
Epoch 4/10, Loss: 0.6267
Epoch 5/10, Loss: 0.6142
Epoch 6/10, Loss: 0.5789
Epoch 7/10, Loss: 0.5848
Epoch 8/10, Loss: 0.5961
Epoch 9/10, Loss: 0.5453
Epoch 10/10, Loss: 0.5440

--- Evaluation Results ---
Accuracy: 0.5714
F1 Score: 0.6897
ROC-AUC:  0.5278
Confusion Matrix:
[[ 2  7]
 [ 2 10]]


In [17]:
model_rn18_ft = models.resnet18(weights='DEFAULT')
model_rn18_ft.fc = nn.Linear(model_rn18_ft.fc.in_features, 1)
model_rn18_ft = model_rn18_ft.to(DEVICE)

# Use a smaller learning rate for fine-tuning to avoid wrecking pretrained weights
optimizer = optim.Adam(model_rn18_ft.parameters(), lr=1e-5)
print("Training Pretrained ResNet-18 (Fine-tuned)...")
train_model(model_rn18_ft, criterion, optimizer)
evaluate_model(model_rn18_ft)

Training Pretrained ResNet-18 (Fine-tuned)...
Epoch 1/10, Loss: 0.8227
Epoch 2/10, Loss: 0.7436
Epoch 3/10, Loss: 0.6857
Epoch 4/10, Loss: 0.6349
Epoch 5/10, Loss: 0.5918
Epoch 6/10, Loss: 0.7795
Epoch 7/10, Loss: 0.6922
Epoch 8/10, Loss: 0.4850
Epoch 9/10, Loss: 0.4631
Epoch 10/10, Loss: 0.4435

--- Evaluation Results ---
Accuracy: 0.6190
F1 Score: 0.5000
ROC-AUC:  0.7222
Confusion Matrix:
[[9 0]
 [8 4]]


In [18]:
model_vit_frozen = models.vit_b_16(weights='DEFAULT')
for param in model_vit_frozen.parameters():
    param.requires_grad = False

# ViT head is called 'heads'
model_vit_frozen.heads = nn.Sequential(nn.Linear(model_vit_frozen.heads[0].in_features, 1))
model_vit_frozen = model_vit_frozen.to(DEVICE)

optimizer = optim.Adam(model_vit_frozen.heads.parameters(), lr=LEARNING_RATE)
print("Training Pretrained ViT-B/16 (Frozen)...")
train_model(model_vit_frozen, criterion, optimizer)
evaluate_model(model_vit_frozen)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\Colton/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth


100.0%


Training Pretrained ViT-B/16 (Frozen)...
Epoch 1/10, Loss: 0.7084
Epoch 2/10, Loss: 0.6469
Epoch 3/10, Loss: 0.6065
Epoch 4/10, Loss: 0.5724
Epoch 5/10, Loss: 0.5497
Epoch 6/10, Loss: 0.5313
Epoch 7/10, Loss: 0.5070
Epoch 8/10, Loss: 0.4877
Epoch 9/10, Loss: 0.4689
Epoch 10/10, Loss: 0.4500

--- Evaluation Results ---
Accuracy: 0.6190
F1 Score: 0.7333
ROC-AUC:  0.5370
Confusion Matrix:
[[ 2  7]
 [ 1 11]]


In [19]:
model_vit_ft = models.vit_b_16(weights='DEFAULT')
model_vit_ft.heads = nn.Sequential(nn.Linear(model_vit_ft.heads[0].in_features, 1))
model_vit_ft = model_vit_ft.to(DEVICE)

# Very small learning rate for ViT fine-tuning
optimizer = optim.Adam(model_vit_ft.parameters(), lr=1e-6)
print("Training Pretrained ViT-B/16 (Fine-tuned)...")
train_model(model_vit_ft, criterion, optimizer)
evaluate_model(model_vit_ft)

Training Pretrained ViT-B/16 (Fine-tuned)...
Epoch 1/10, Loss: 0.7099
Epoch 2/10, Loss: 0.6868
Epoch 3/10, Loss: 0.6719
Epoch 4/10, Loss: 0.6603
Epoch 5/10, Loss: 0.6453
Epoch 6/10, Loss: 0.6368
Epoch 7/10, Loss: 0.6225
Epoch 8/10, Loss: 0.6140
Epoch 9/10, Loss: 0.6020
Epoch 10/10, Loss: 0.5908

--- Evaluation Results ---
Accuracy: 0.5714
F1 Score: 0.7097
ROC-AUC:  0.5741
Confusion Matrix:
[[ 1  8]
 [ 1 11]]


In [20]:
def collect_metrics(model, model_name, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy().astype(int))

    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    
    # Flatten confusion matrix for the CSV (TN, FP, FN, TP)
    tn, fp, fn, tp = cm.ravel()
    
    return {
        "Model": model_name,
        "Accuracy": acc,
        "F1_Score": f1,
        "ROC_AUC": roc,
        "True_Negative": tn,
        "False_Positive": fp,
        "False_Negative": fn,
        "True_Positive": tp
    }

# 1. Create a list of your models and their display names
model_list = [
    (model_cnn, "Custom CNN"),
    (model_rn18_scratch, "ResNet-18 (Learned Weights)"),
    (model_rn18_frozen, "ResNet-18 (Pretrained-Frozen)"),
    (model_rn18_ft, "ResNet-18 (Pretrained-FineTuned)"),
    (model_vit_frozen, "ViT-B/16 (Pretrained-Frozen)"),
    (model_vit_ft, "ViT-B/16 (Pretrained-FineTuned)")
]

# 2. Loop through and gather metrics
results_data = []
for model_obj, name in model_list:
    print(f"Calculating metrics for {name}...")
    metrics = collect_metrics(model_obj, name, test_loader)
    results_data.append(metrics)

# 3. Create DataFrame and Save
results_df = pd.DataFrame(results_data)
results_df.to_csv('model_performance_results.csv', index=False)

print("\nSuccess! Results saved to 'model_performance_results.csv'")
results_df # Display the table in your notebook

Calculating metrics for Custom CNN...
Calculating metrics for ResNet-18 (Learned Weights)...
Calculating metrics for ResNet-18 (Pretrained-Frozen)...
Calculating metrics for ResNet-18 (Pretrained-FineTuned)...
Calculating metrics for ViT-B/16 (Pretrained-Frozen)...
Calculating metrics for ViT-B/16 (Pretrained-FineTuned)...

Success! Results saved to 'model_performance_results.csv'


,Model,Accuracy,F1_Score,ROC_AUC,True_Negative,False_Positive,False_Negative,True_Positive
0,Custom CNN,0.619048,0.750000,0.620370,1,8,0,12
1,ResNet-18 (Learned Weights),0.523810,0.687500,0.388889,0,9,1,11
2,ResNet-18 (Pretrained-Frozen),0.571429,0.689655,0.527778,2,7,2,10
3,ResNet-18 (Pretrained-FineTuned),0.619048,0.500000,0.666667,9,0,8,4
4,ViT-B/16 (Pretrained-Frozen),0.619048,0.733333,0.537037,2,7,1,11
5,ViT-B/16 (Pretrained-FineTuned),0.571429,0.709677,0.578704,1,8,1,11
